In [11]:
!pip install -q --upgrade google-cloud-vision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 6.0 MB/s eta 0:00:00


In [1]:
%env "# REMOVED: GCP service account key not needed for local execution"

In [8]:
import pandas as pd

df = pd.read_csv('../data/2023-10-12-Final-Instaloader-Crowdtangle-Merged-w-filenames-and-original-image-filenames-and-faces.csv', index_col=0)
df.drop(columns=['Faces'], inplace=True)

In [9]:
df.head()

,username,shortcode,original_file,corresponding_image,image
0,afd.bund,CUQWJvuF_aS,./afd.bund/2021-09-25_18-55-01_UTC.json,CUQWJvuF_aS_0.jpg,afd.bund/2021-09-25_18-55-01_UTC.jpg
1,afd.bund,CUQCd4VqbQS,./afd.bund/2021-09-25_16-02-49_UTC.json,CUQCd4VqbQS_0.jpg,afd.bund/2021-09-25_16-02-49_UTC.jpg
2,afd.bund,CUP0YxGlW6L,./afd.bund/2021-09-25_14-00-00_UTC.json,CUP0YxGlW6L_0.jpg,afd.bund/2021-09-25_14-00-00_UTC.jpg
3,afd.bund,CUPf-RSN6l3,./afd.bund/2021-09-25_11-01-21_UTC.json,CUPf-RSN6l3_0.jpg,afd.bund/2021-09-25_11-01-21_UTC_0.jpg
4,afd.bund,CUPf-RSN6l3,./afd.bund/2021-09-25_11-01-21_UTC.json,CUPf-RSN6l3_1.jpg,afd.bund/2021-09-25_11-01-21_UTC_1.jpg


In [ ]:
#@title Sign Image Links
from google.cloud import storage
import datetime

client = storage.Client.from_service_account_json("# REMOVED: GCP service account key not needed for local execution")

def signed_url(url, days=7):
  # Removing the 'gs://' prefix
  stripped_url = url.replace("gs://", "")

  # Splitting at the first '/'
  split_url = stripped_url.split("/", 1)

  bucket_name = "ig-politik-stories"
  blob_name = stripped_url

  bucket = client.bucket(bucket_name)
  blob = bucket.blob(blob_name)


  url = blob.generate_signed_url(
        version="v4",
        # This URL is valid for 15 minutes
        expiration=datetime.timedelta(days=days),
        # Allow GET requests using this URL.
        method="GET")
  return url

df['storage_url'] = df['image'].apply(signed_url)

In [ ]:
import pandas as pd
from google.cloud import vision
from tqdm.notebook import tqdm


def localize_objects_uri(df):
    """Localize objects in images on Google Cloud Storage using a dataframe.

    Args:
    df: A dataframe containing a column 'image' with paths to the files.

    Returns:
    DataFrame with columns: image, object_name, confidence, and vertices
    """
    client = vision.ImageAnnotatorClient()
    results = []

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        uri = "../data/" + row['image']
        image = vision.Image()
        image.source.image_uri = uri

        objects = client.object_localization(image=image).localized_object_annotations

        for object_ in objects:
            vertices = [{"x": vertex.x, "y": vertex.y} for vertex in object_.bounding_poly.normalized_vertices]
            result = {
                "image": row['image'],
                "object_name": object_.name,
                "confidence": object_.score,
                "vertices": vertices
            }
            results.append(result)

    return pd.DataFrame(results)

output_df = localize_objects_uri(df)
output_df.to_csv('../data/2023-10-02-Post-Google-Cloud-Vision-Object-Detection.csv')

In [14]:
output_df

,image,object_name,confidence,vertices
0,afd.bund/2021-09-25_18-55-01_UTC.jpg,Person,0.848779,"[{'x': 0.04946938157081604, 'y': 0.37492620944..."
1,afd.bund/2021-09-25_18-55-01_UTC.jpg,Tie,0.844848,"[{'x': 0.33461809158325195, 'y': 0.55930507183..."
2,afd.bund/2021-09-25_18-55-01_UTC.jpg,Suit,0.652835,"[{'x': 0.051026687026023865, 'y': 0.4214123189..."
3,afd.bund/2021-09-25_18-55-01_UTC.jpg,Glasses,0.553569,"[{'x': 0.19909803569316864, 'y': 0.37253710627..."
4,afd.bund/2021-09-25_16-02-49_UTC.jpg,Person,0.916204,"[{'x': 0.32995298504829407, 'y': 0.00841491762..."
...,...,...,...,...
5512,spdde/2021-09-25_10-04-18_UTC.jpg,Person,0.885907,"[{'x': 0.24210470914840698, 'y': 0.40299639105..."
5513,spdde/2021-09-25_10-04-18_UTC.jpg,Person,0.863928,"[{'x': 0.3874337673187256, 'y': 0.385825783014..."
5514,spdde/2021-09-25_10-04-18_UTC.jpg,Person,0.835010,"[{'x': 0.15837237238883972, 'y': 0.43293869495..."
5515,spdde/2021-09-25_10-04-18_UTC.jpg,Clothing,0.790697,"[{'x': 0.10292163491249084, 'y': 0.63410794734..."
